# 🎭 eidcloud-api-mock: Quickstart & Chaos Sandbox

Welcome to the interactive notebook for **eidcloud-api-mock** — the pure PHP 8.2+ schema-driven API Mock Server and Sandbox engine.

In this notebook, we will:
1. Clone repository and install/verify PHP 8.2+ environment.
2. Parse OpenAPI specs, launch the Mock Server in background, and test stateful CRUD operations (`POST` -> `GET`).
3. Simulate chaos engineering (network latency injection and intermittent 5xx HTTP error spikes).

In [ ]:
# Cell 1: Environment Setup & OpenAPI Spec Generation
!apt-get update -qq && apt-get install -y -qq php-cli
!git clone https://github.com/eidcloud/eidcloud-api-mock.git /content/eidcloud-api-mock || true
%cd /content/eidcloud-api-mock

# Verify pure PHP test suite
!php tests/run_tests.php

In [ ]:
# Cell 2: Stateful Mock Engine & Chaos Simulation in Action
import subprocess, time, requests, json

# 1. Launch Mock Server with 150ms latency and 20% chaos error injection
server_proc = subprocess.Popen([
    'php', 'bin/eidcloud-mock', 'serve', 'examples/openapi.json',
    '--host=127.0.0.1', '--port=8080', '--delay=150ms', '--error-rate=20%'
], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(1.5)

base_url = 'http://127.0.0.1:8080'
print('🚀 eidcloud-api-mock server listening on', base_url)

# 2. Test Stateful POST /users
new_user = {'name': 'Eng. Shadi AL-Hasan', 'email': 'shadi@eidcloud.io', 'role': 'admin'}
resp_create = requests.post(f'{base_url}/users', json=new_user)
print(f'\n[POST /users] Status: {resp_create.status_code} | Latency: {resp_create.headers.get("X-Mock-Delay-Ms")}ms')
created_data = resp_create.json()
print('Created Entity:', json.dumps(created_data, indent=2))

user_id = created_data.get('id', 1)

# 3. Test Stateful GET /users/{id} to verify memory persistence
resp_get = requests.get(f'{base_url}/users/{user_id}')
print(f'\n[GET /users/{user_id}] Status: {resp_get.status_code}')
print('Retrieved State:', json.dumps(resp_get.json(), indent=2))

# 4. Simulate Latency & Chaos Spikes
print('\n🔥 Executing 10 concurrent requests to observe simulated latency & chaos errors...')
for i in range(10):
    t0 = time.time()
    res = requests.get(f'{base_url}/users')
    elapsed = int((time.time() - t0) * 1000)
    is_chaos = res.headers.get('X-Mock-Chaos', 'false')
    print(f'  Request #{i+1:02d}: HTTP {res.status_code} | Took {elapsed}ms | Chaos Triggered: {is_chaos}')

server_proc.terminate()
print('\n✅ Sandbox run complete!')